# SR-MAMBA-AMC: Official Mamba, Kaggle T4 ×2

This notebook uses the official `mamba_ssm.Mamba` CUDA implementation only. It does not substitute a Conv1D fallback. Use a fresh Kaggle GPU session and enable Internet before running the install cell.

Memory safety: window 1024, global batch 8, gradient accumulation 4, DataLoader workers 0, and pretraining disabled.

## 1. Install official Mamba and configure training

In [5]:
# Run this in a fresh Kaggle GPU session. If Kaggle asks for a kernel restart after installation, restart then Run All.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'causal-conv1d', 'mamba-ssm'])

0

In [6]:
from pathlib import Path
from fractions import Fraction
from collections import Counter
import json, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import resample_poly
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from mamba_ssm import Mamba
assert torch.cuda.is_available(), 'This official-Mamba notebook requires a Kaggle CUDA GPU.'
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.benchmark=True
CONFIG={'window_size':1024,'patch_size':8,'embedding_dim':256,'mamba_blocks':8,'d_state':16,'conv_kernel':4,'mamba_expand':2,'transformer_blocks':2,'attention_heads':4,'dropout':.20,'target_sps':8.,'batch_size':8,'accumulate_steps':4,'epochs':20,'learning_rate':1e-4,'weight_decay':2e-4,'gradient_clip':1.,'patience':10,'label_smoothing':.1,'num_workers':0,'feature_stats_samples':1024,'noise_snr_range':(-10,20),'cfo_probability':.25,'max_cfo_cycles_per_sample':.01}
device=torch.device('cuda'); gpu_count=torch.cuda.device_count(); print('Official Mamba | GPUs:',gpu_count,[torch.cuda.get_device_name(i) for i in range(gpu_count)])
OUTPUT_ROOT=Path('/kaggle/working'); REPORTS=OUTPUT_ROOT/'reports'; FIGURES=REPORTS/'figures'; CURVES=REPORTS/'training_curves'; MODELS=OUTPUT_ROOT/'models'
for p in (REPORTS,FIGURES,CURVES,MODELS): p.mkdir(parents=True,exist_ok=True)
roots=[Path('/kaggle/input/datasets/henypatel26/nexgenninjas/complete_dataset'),Path('/kaggle/input/nexgenninjas/complete_dataset')]; roots += [p.parent for p in Path('/kaggle/input').rglob('recordings.csv') if p.parent.name=='metadata']; DATASET_ROOT=next((p for p in roots if (p/'metadata/recordings.csv').exists()),None); assert DATASET_ROOT, 'Attach complete_dataset.'
CLASSES=['OOK','PAM','2FSK','4FSK','CPFSK','GMSK','BPSK','QPSK','8PSK','16QAM','64QAM','AM','FM','NOISE']; CLASS_ID={c:i for i,c in enumerate(CLASSES)}
df=pd.read_csv(DATASET_ROOT/'metadata/recordings.csv',low_memory=False); label_col=next(c for c in ('primary_modulation_class','modulation_class','label') if c in df); path_col=next(c for c in ('iq_file','file_path','path','file_name') if c in df)
def resolve(v):
    s=str(v).replace('\\','/'); return DATASET_ROOT/s.split('complete_dataset/',1)[1] if 'complete_dataset/' in s else DATASET_ROOT/s
df['label']=df[label_col].astype(str).str.upper(); df['_path']=df[path_col].map(resolve); df['split']=df['split'].astype(str).str.lower(); snr_col=next((c for c in ('primary_snr_db','snr_db','measured_snr_db') if c in df),None); sps_col='samples_per_symbol' if 'samples_per_symbol' in df else None
assert not(set(CLASSES)-set(df.label)); splits={k:df[(df.label.isin(CLASSES))&(df.split==k)].copy() for k in ('train','validation','test')}; assert all(len(x) for x in splits.values())
for k,v in splits.items(): print(k,len(v),v.label.value_counts().reindex(CLASSES).to_dict())

Official Mamba | GPUs: 2 ['Tesla T4', 'Tesla T4']
train 69972 {'OOK': 4998, 'PAM': 4998, '2FSK': 4998, '4FSK': 4998, 'CPFSK': 4998, 'GMSK': 4998, 'BPSK': 4998, 'QPSK': 4998, '8PSK': 4998, '16QAM': 4998, '64QAM': 4998, 'AM': 4998, 'FM': 4998, 'NOISE': 4998}
validation 14994 {'OOK': 1071, 'PAM': 1071, '2FSK': 1071, '4FSK': 1071, 'CPFSK': 1071, 'GMSK': 1071, 'BPSK': 1071, 'QPSK': 1071, '8PSK': 1071, '16QAM': 1071, '64QAM': 1071, 'AM': 1071, 'FM': 1071, 'NOISE': 1071}
test 14994 {'OOK': 1071, 'PAM': 1071, '2FSK': 1071, '4FSK': 1071, 'CPFSK': 1071, 'GMSK': 1071, 'BPSK': 1071, 'QPSK': 1071, '8PSK': 1071, '16QAM': 1071, '64QAM': 1071, 'AM': 1071, 'FM': 1071, 'NOISE': 1071}


## 2. IQ pipeline and train-only feature normalization

In [7]:
def load_iq(r):
    p=Path(r['_path']); dtype=str(r.get('true_dtype','complex64')).lower(); layout=str(r.get('iq_layout_variant','interleaved_iq')).lower()
    if p.suffix.lower()=='.wav':
        _,x=wavfile.read(p); return x[:,0].astype(np.float32)+1j*x[:,1].astype(np.float32)
    if 'complex64' in dtype:return np.fromfile(p,np.complex64)
    if 'complex128' in dtype:return np.fromfile(p,np.complex128).astype(np.complex64)
    dt=np.int8 if 'int8' in dtype else np.int16 if 'int16' in dtype else np.float32; x=np.fromfile(p,dt).astype(np.float32); sc=float(np.iinfo(dt).max) if np.issubdtype(dt,np.integer) else 1.; return ((x[:len(x)//2]+1j*x[len(x)//2:]) if 'block' in layout else x[0::2]+1j*x[1::2])/sc
def norm(z):
    z=np.asarray(z,np.complex64); z=z-z.mean(); return z/np.sqrt(max(float(np.mean(np.abs(z)**2)),1e-12))
def crop(z,train=False):
    z=norm(z); n=CONFIG['window_size']; z=np.pad(z,(0,max(0,n-len(z)))); start=np.random.randint(0,len(z)-n+1) if train and len(z)>n else max(0,(len(z)-n)//2); return z[start:start+n]
def augment(z):
    z=z.copy()
    if np.random.rand()<.5:
        snr=np.random.uniform(*CONFIG['noise_snr_range']); p=np.mean(np.abs(z)**2); z+=np.sqrt(p/(2*10**(snr/10)))*(np.random.randn(len(z))+1j*np.random.randn(len(z)))
    if np.random.rand()<.25:z=np.conj(z)
    if np.random.rand()<.2:z=z[::-1].copy()
    if np.random.rand()<.5:z*=np.exp(1j*np.random.choice([0,np.pi/2,np.pi,3*np.pi/2]))
    if np.random.rand()<CONFIG['cfo_probability']:z*=np.exp(2j*np.pi*np.random.uniform(-CONFIG['max_cfo_cycles_per_sample'],CONFIG['max_cfo_cycles_per_sample'])*np.arange(len(z)))
    return norm(z)
def features(z):
    z=norm(z); p=np.mean(np.abs(z)**2); m20=np.mean(z*z); e=z[:len(z)//2*2]; h=(e[::2]+e[1::2])/np.sqrt(2); d=(e[::2]-e[1::2])/np.sqrt(2); return np.asarray([abs(m20),abs(np.mean(z**4)-3*m20*m20),abs(np.mean(z**3*np.conj(z))-3*m20*p),abs(np.mean(np.abs(z)**4)-abs(m20)**2-2*p*p),abs(np.mean(z**6)),abs(np.mean(z**5*np.conj(z))),abs(np.mean(z**4*np.conj(z)**2)),abs(np.mean(z**3*np.conj(z)**3)),abs(np.mean(z**8)),np.mean(np.abs(h)),np.var(np.abs(h)),10*np.log10(p/(np.var(d)+1e-12))],np.float32)
def canonical(z,sps):
    if not np.isfinite(sps) or sps<=0:return crop(z)
    r=Fraction(CONFIG['target_sps']/float(sps)).limit_denominator(100); return crop(resample_poly(z,r.numerator,r.denominator).astype(np.complex64))
sample=splits['train'].groupby('label',group_keys=False).head(max(1,CONFIG['feature_stats_samples']//len(CLASSES))); stats=np.stack([features(crop(load_iq(r))) for _,r in tqdm(sample.iterrows(),total=len(sample),desc='Feature stats')]); MEAN=stats.mean(0); STD=np.maximum(stats.std(0),1e-6); np.savez(REPORTS/'feature_scaling.npz',mean=MEAN,std=STD)
class IQDataset(Dataset):
    def __init__(self,frame,train=False):self.frame=frame.reset_index(drop=True);self.train=train
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]; z=crop(load_iq(r),self.train); target=z.copy(); raw=augment(z) if self.train else z; sps=float(r.get(sps_col,np.nan)) if sps_col else np.nan; pack=lambda a:torch.from_numpy(np.stack((a.real,a.imag)).astype(np.float32)); return {'raw':pack(raw),'canon':pack(canonical(raw,sps)),'feat':torch.from_numpy(((features(raw)-MEAN)/STD).astype(np.float32)),'target':pack(target),'label':torch.tensor(CLASS_ID[r.label]),'sps':torch.tensor(sps if np.isfinite(sps) else -1.,dtype=torch.float32)}

Feature stats:   0%|          | 0/1022 [00:00<?, ?it/s]

## 3. Original official bidirectional Mamba model

In [8]:
class OfficialBiMamba(nn.Module):
    def __init__(self,d):
        super().__init__(); self.norm=nn.LayerNorm(d); self.conv=nn.Conv1d(d,d,CONFIG['conv_kernel'],padding=CONFIG['conv_kernel']-1,groups=d); self.gate=nn.Linear(d,d); self.fwd=Mamba(d_model=d,d_state=CONFIG['d_state'],d_conv=CONFIG['conv_kernel'],expand=CONFIG['mamba_expand']); self.bwd=Mamba(d_model=d,d_state=CONFIG['d_state'],d_conv=CONFIG['conv_kernel'],expand=CONFIG['mamba_expand']); self.merge=nn.Linear(2*d,d); self.drop=nn.Dropout(CONFIG['dropout'])
    def forward(self,x):
        h=self.norm(x); h=self.conv(h.transpose(1,2))[...,:h.size(1)].transpose(1,2)*torch.sigmoid(self.gate(h)); f=self.fwd(h); b=torch.flip(self.bwd(torch.flip(h,[1])),[1]); return x+self.drop(self.merge(torch.cat((f,b),-1)))
class FeatureTokens(nn.Module):
    def __init__(self,d):super().__init__();self.c=nn.Sequential(nn.LayerNorm(9),nn.Linear(9,d),nn.GELU());self.h=nn.Sequential(nn.LayerNorm(2),nn.Linear(2,d),nn.GELU());self.s=nn.Sequential(nn.LayerNorm(1),nn.Linear(1,d),nn.GELU())
    def forward(self,x):return torch.stack((self.c(x[:,:9]),self.h(x[:,9:11]),self.s(x[:,11:])),1)
class SRMambaOfficial(nn.Module):
    def __init__(self):
        super().__init__();d=CONFIG['embedding_dim'];t=CONFIG['window_size']//CONFIG['patch_size'];self.patch=nn.Conv1d(2,d,CONFIG['patch_size'],stride=CONFIG['patch_size']);self.pos=nn.Parameter(torch.zeros(1,t,d));self.blocks=nn.ModuleList([OfficialBiMamba(d) for _ in range(CONFIG['mamba_blocks'])]);self.denoise=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),nn.GELU(),nn.Dropout(CONFIG['dropout']),nn.Linear(d,d));layer=nn.TransformerEncoderLayer(d,CONFIG['attention_heads'],4*d,CONFIG['dropout'],batch_first=True,norm_first=True);self.transformer=nn.TransformerEncoder(layer,CONFIG['transformer_blocks']);self.tokens=FeatureTokens(d);self.cross=nn.MultiheadAttention(d,CONFIG['attention_heads'],batch_first=True);self.fuse=nn.Sequential(nn.LayerNorm(2*d),nn.Linear(2*d,d),nn.GELU(),nn.Dropout(CONFIG['dropout']));self.classifier=nn.Linear(d,14);self.sps=nn.Linear(d,2);self.decoder=nn.Linear(d,2*CONFIG['window_size'])
    def encode(self,x):
        h=self.patch(x).transpose(1,2)+self.pos
        for b in self.blocks:h=b(h)
        return self.transformer(h+self.denoise(h)).mean(1)
    def forward(self,raw,canon,feat):
        a,b=self.encode(raw),self.encode(canon);n=(a+b)/2;att,_=self.cross(n.unsqueeze(1),self.tokens(feat),self.tokens(feat));h=self.fuse(torch.cat((n,att.squeeze(1)),1));s=self.sps(h);return {'logits':self.classifier(h),'sps_mu':s[:,0],'sps_logvar':s[:,1].clamp(-8,8),'recon':self.decoder(h).view(-1,2,CONFIG['window_size']),'raw':a,'canon':b}
base=SRMambaOfficial().to(device);model=nn.DataParallel(base) if gpu_count>1 else base;print('Official Mamba parameters:',f'{sum(p.numel() for p in base.parameters()):,}','| DataParallel:',gpu_count>1)
def loss_fn(out,b):
    cls=F.cross_entropy(out['logits'],b['label'],label_smoothing=CONFIG['label_smoothing']);valid=b['sps']>0;sps=F.gaussian_nll_loss(out['sps_mu'][valid],torch.log2(b['sps'][valid]),out['sps_logvar'][valid].exp()) if valid.any() else cls.new_zeros(());view=F.mse_loss(F.normalize(out['raw'],dim=1),F.normalize(out['canon'],dim=1));recon=F.mse_loss(out['recon'],b['target']);return cls+.2*sps+.15*view+.1*recon

/tmp/ipykernel_58/162523223.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  super().__init__();d=CONFIG['embedding_dim'];t=CONFIG['window_size']//CONFIG['patch_size'];self.patch=nn.Conv1d(2,d,CONFIG['patch_size'],stride=CONFIG['patch_size']);self.pos=nn.Parameter(torch.zeros(1,t,d));self.blocks=nn.ModuleList([OfficialBiMamba(d) for _ in range(CONFIG['mamba_blocks'])]);self.denoise=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),nn.GELU(),nn.Dropout(CONFIG['dropout']),nn.Linear(d,d));layer=nn.TransformerEncoderLayer(d,CONFIG['attention_heads'],4*d,CONFIG['dropout'],batch_first=True,norm_first=True);self.transformer=nn.TransformerEncoder(layer,CONFIG['transformer_blocks']);self.tokens=FeatureTokens(d);self.cross=nn.MultiheadAttention(d,CONFIG['attention_heads'],batch_first=True);self.fuse=nn.Sequential(nn.LayerNorm(2*d),nn.Linear(2*d,d),nn.GELU(),nn.Dropout(CONFIG['dropout']));self.classifier=nn.Linear(d,14);s

Official Mamba parameters: 11,274,024 | DataParallel: True


## 4. Stable official-Mamba training and evaluation

In [ ]:
def make_loader(frame,train=False):return DataLoader(IQDataset(frame,train),batch_size=CONFIG['batch_size'],shuffle=train,num_workers=0,pin_memory=True,drop_last=train)
val_loader=make_loader(splits['validation']);test_loader=make_loader(splits['test']);opt=torch.optim.AdamW(model.parameters(),lr=CONFIG['learning_rate'],weight_decay=CONFIG['weight_decay']);schedule=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CONFIG['epochs']);scaler=torch.cuda.amp.GradScaler()
def move(b):return {k:v.to(device,non_blocking=True) for k,v in b.items()}
def unwrap():return (model.module if isinstance(model,nn.DataParallel) else model).state_dict()
@torch.no_grad()
def evaluate(loader):
    model.eval();y=[];p=[];total=0
    for b in loader:
        b=move(b);out=model(b['raw'],b['canon'],b['feat']);loss=loss_fn(out,b);total+=loss.item()*len(b['label']);y.extend(b['label'].cpu().numpy());p.extend(out['logits'].argmax(1).cpu().numpy())
    return total/len(y),accuracy_score(y,p),f1_score(y,p,average='macro',zero_division=0)
history=[];best=-1.;stale=0;best_epoch=0
for epoch in range(1,CONFIG['epochs']+1):
    stage=min(3,(epoch-1)*4//CONFIG['epochs']);threshold=[15,5,-5,None][stage] if snr_col else None;frame=splits['train'] if threshold is None else splits['train'][pd.to_numeric(splits['train'][snr_col],errors='coerce')>=threshold];frame=frame if len(frame)>=CONFIG['batch_size'] else splits['train']
    model.train();opt.zero_grad(set_to_none=True);total=correct=seen=0
    for step,b in enumerate(tqdm(make_loader(frame,True),desc=f'Epoch {epoch}/{CONFIG["epochs"]}'),1):
        b=move(b)
        with torch.cuda.amp.autocast():out=model(b['raw'],b['canon'],b['feat']);loss=loss_fn(out,b)/CONFIG['accumulate_steps']
        scaler.scale(loss).backward();total+=loss.item()*CONFIG['accumulate_steps']*len(b['label']);correct+=(out['logits'].argmax(1)==b['label']).sum().item();seen+=len(b['label'])
        if step%CONFIG['accumulate_steps']==0:scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG['gradient_clip']);scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
    if step%CONFIG['accumulate_steps']!=0:scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG['gradient_clip']);scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
    vl,va,vf=evaluate(val_loader);row={'epoch':epoch,'train_loss':total/seen,'train_accuracy':correct/seen,'val_loss':vl,'val_accuracy':va,'val_macro_f1':vf};history.append(row);print(row);schedule.step()
    ck={'model_state_dict':unwrap(),'epoch':epoch,'best_validation_macro_f1':max(best,vf),'class_names':CLASSES,'configuration':CONFIG,'feature_mean':MEAN.tolist(),'feature_std':STD.tolist(),'history':history,'mamba_backend':'official mamba_ssm'};torch.save(ck,MODELS/'sr_mamba_official_last.pt')
    if vf>best:best=vf;best_epoch=epoch;stale=0;torch.save(ck,MODELS/'sr_mamba_official_best.pt');torch.save(ck,OUTPUT_ROOT/'model.pt')
    else:stale+=1
    if stale>=CONFIG['patience']:print('Early stop on validation Macro-F1.');break
history_df=pd.DataFrame(history);history_df.to_csv(REPORTS/'training_history.csv',index=False);history_df.plot(x='epoch',y=['train_loss','val_loss'],marker='o');plt.tight_layout();plt.savefig(CURVES/'loss_curve.png',dpi=160);plt.show()
best_ck=torch.load(MODELS/'sr_mamba_official_best.pt',map_location=device,weights_only=False);(model.module if isinstance(model,nn.DataParallel) else model).load_state_dict(best_ck['model_state_dict']);model.eval();y=[];logits=[]
with torch.no_grad():
    for b in tqdm(test_loader,desc='test'):b=move(b);out=model(b['raw'],b['canon'],b['feat']);y.extend(b['label'].cpu().numpy());logits.extend(out['logits'].cpu().numpy())
y=np.asarray(y);prob=F.softmax(torch.tensor(np.asarray(logits)),1).numpy();pred=prob.argmax(1);macro=precision_recall_fscore_support(y,pred,average='macro',zero_division=0);metrics={'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),'macro_f1':macro[2]};(REPORTS/'test_metrics.json').write_text(json.dumps(metrics,indent=2));pd.DataFrame(classification_report(y,pred,target_names=CLASSES,output_dict=True,zero_division=0)).T.to_csv(REPORTS/'classification_report.csv');cm=confusion_matrix(y,pred);pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(REPORTS/'confusion_matrix.csv');print('COMPLETE',metrics,'best epoch',best_epoch,'checkpoint',MODELS/'sr_mamba_official_best.pt')

/tmp/ipykernel_58/3173266122.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  val_loader=make_loader(splits['validation']);test_loader=make_loader(splits['test']);opt=torch.optim.AdamW(model.parameters(),lr=CONFIG['learning_rate'],weight_decay=CONFIG['weight_decay']);schedule=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CONFIG['epochs']);scaler=torch.cuda.amp.GradScaler()


Epoch 1/20:   0%|          | 0/2320 [00:00<?, ?it/s]

/tmp/ipykernel_58/3173266122.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():out=model(b['raw'],b['canon'],b['feat']);loss=loss_fn(out,b)/CONFIG['accumulate_steps']


{'epoch': 1, 'train_loss': 1.951024145735749, 'train_accuracy': 0.3652478448275862, 'val_loss': 1.850740644348865, 'val_accuracy': 0.44757903161264506, 'val_macro_f1': 0.4119150482832941}


Epoch 2/20:   0%|          | 0/2320 [00:00<?, ?it/s]

...

Epoch 8/20:   0%|          | 0/4641 [00:00<?, ?it/s]

/tmp/ipykernel_58/3173266122.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():out=model(b['raw'],b['canon'],b['feat']);loss=loss_fn(out,b)/CONFIG['accumulate_steps']


{'epoch': 8, 'train_loss': 0.8630652020227239, 'train_accuracy': 0.7434550743374273, 'val_loss': 1.2416879878261018, 'val_accuracy': 0.6465919701213819, 'val_macro_f1': 0.6161561976809588}


Epoch 9/20:   0%|          | 0/4641 [00:00<?, ?it/s]

/tmp/ipykernel_58/3173266122.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():out=model(b['raw'],b['canon'],b['feat']);loss=loss_fn(out,b)/CONFIG['accumulate_steps']


## 5. Detailed training and test-report figures

Run this cell after the training cell. It saves graphs and detailed CSV reports under `/kaggle/working/reports/`.

In [1]:
# Run after training/evaluation. `y`, `pred`, `prob`, `history`, and `splits` are created above.
from matplotlib.ticker import PercentFormatter
plt.style.use('seaborn-v0_8-whitegrid')
FIGURES = REPORTS / 'figures'; FIGURES.mkdir(parents=True, exist_ok=True)

# Training curves: early stopping selects the maximum validation macro-F1.
hist = pd.read_csv(REPORTS / 'training_history.csv')
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
ax[0].plot(hist.epoch, hist.train_loss, 'o-', label='Train'); ax[0].plot(hist.epoch, hist.val_loss, 'o-', label='Validation'); ax[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy'); ax[0].legend()
ax[1].plot(hist.epoch, hist.train_accuracy, 'o-', label='Train'); ax[1].plot(hist.epoch, hist.val_accuracy, 'o-', label='Validation'); ax[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy', ylim=(0,1)); ax[1].yaxis.set_major_formatter(PercentFormatter(1)); ax[1].legend()
best_i = hist.val_macro_f1.idxmax(); ax[2].plot(hist.epoch, hist.val_macro_f1, 'o-', color='#2ca02c'); ax[2].scatter(hist.loc[best_i, 'epoch'], hist.loc[best_i, 'val_macro_f1'], color='crimson', zorder=3, label=f"Best: epoch {int(hist.loc[best_i, 'epoch'])}"); ax[2].set(title='Validation macro-F1', xlabel='Epoch', ylabel='Macro-F1', ylim=(0,1)); ax[2].yaxis.set_major_formatter(PercentFormatter(1)); ax[2].legend()
fig.suptitle('SR-MAMBA-AMC training history', y=1.03); fig.tight_layout(); fig.savefig(FIGURES/'training_curves_detailed.png', dpi=220, bbox_inches='tight'); plt.show()

# Detailed confusion matrices: left is count, right is recall (%) for every true modulation class.
cm_counts = confusion_matrix(y, pred, labels=np.arange(len(CLASSES)))
cm_pct = confusion_matrix(y, pred, labels=np.arange(len(CLASSES)), normalize='true') * 100
pd.DataFrame(cm_counts, index=CLASSES, columns=CLASSES).to_csv(REPORTS/'confusion_matrix_counts.csv')
pd.DataFrame(cm_pct, index=CLASSES, columns=CLASSES).to_csv(REPORTS/'confusion_matrix_row_percent.csv')
fig, ax = plt.subplots(1, 2, figsize=(23, 9))
for a, matrix, title, fmt, cmap in [(ax[0], cm_counts, 'Test examples (counts)', 'd', 'Blues'), (ax[1], cm_pct, 'Per-true-class recall (%)', '.1f', 'YlOrRd')]:
    im=a.imshow(matrix, cmap=cmap); fig.colorbar(im, ax=a, fraction=.046, pad=.04); a.set(xticks=np.arange(len(CLASSES)), yticks=np.arange(len(CLASSES)), xticklabels=CLASSES, yticklabels=CLASSES, xlabel='Predicted modulation', ylabel='True modulation', title=title); a.tick_params(axis='x', rotation=45, labelsize=9); a.tick_params(axis='y', labelsize=9)
    cutoff=matrix.max()*.58
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)): a.text(j,i,format(matrix[i,j],fmt),ha='center',va='center',fontsize=7,color='white' if matrix[i,j]>cutoff else 'black')
fig.tight_layout(); fig.savefig(FIGURES/'confusion_matrices_detailed.png', dpi=250, bbox_inches='tight'); plt.show()

# Class-level precision, recall, F1 and sample support.
per_class=pd.DataFrame(classification_report(y,pred,labels=np.arange(len(CLASSES)),target_names=CLASSES,output_dict=True,zero_division=0)).T.loc[CLASSES]; per_class.to_csv(REPORTS/'per_class_metrics.csv')
fig, ax=plt.subplots(figsize=(15,6)); x=np.arange(len(CLASSES)); w=.25
for shift, metric, color in [(-w,'precision','#4C78A8'),(0,'recall','#F58518'),(w,'f1-score','#54A24B')]: ax.bar(x+shift,per_class[metric],w,label=metric.title(),color=color)
ax.set(xticks=x,xticklabels=CLASSES,ylim=(0,1.05),ylabel='Score',title='Per-class test precision, recall and F1'); ax.yaxis.set_major_formatter(PercentFormatter(1)); ax.legend(ncol=3); plt.xticks(rotation=35,ha='right'); fig.tight_layout(); fig.savefig(FIGURES/'per_class_metrics.png',dpi=220,bbox_inches='tight'); plt.show()

# SNR robustness graph, if test metadata contains an SNR column.
if snr_col and snr_col in splits['test'].columns:
    test_snr=pd.to_numeric(splits['test'][snr_col],errors='coerce').to_numpy(); rows=[]
    for lo in range(-20,31,5):
        mask=np.isfinite(test_snr)&(test_snr>=lo)&(test_snr<lo+5)
        if mask.any(): rows.append({'snr_mid':lo+2.5,'snr_bin':f'{lo} to {lo+5}','samples':int(mask.sum()),'accuracy':accuracy_score(y[mask],pred[mask]),'macro_f1':f1_score(y[mask],pred[mask],average='macro',zero_division=0)})
    snr_results=pd.DataFrame(rows)
    if len(snr_results):
        snr_results.to_csv(REPORTS/'snr_performance.csv',index=False); fig, a=plt.subplots(figsize=(12,5)); a.plot(snr_results.snr_mid,snr_results.accuracy,'o-',label='Accuracy'); a.plot(snr_results.snr_mid,snr_results.macro_f1,'s-',label='Macro-F1'); a.set(xlabel='SNR (dB)',ylabel='Score',ylim=(0,1.05),title='Test performance by SNR'); a.yaxis.set_major_formatter(PercentFormatter(1)); a.legend(); fig.tight_layout(); fig.savefig(FIGURES/'snr_performance.png',dpi=220,bbox_inches='tight'); plt.show()

# Confidence reliability curve; helpful for judging prediction trustworthiness.
conf=prob.max(1); correct=(pred==y).astype(float); edges=np.linspace(0,1,11); idx=np.clip(np.digitize(conf,edges,right=True)-1,0,9)
cal=pd.DataFrame([{'bin_low':edges[i],'bin_high':edges[i+1],'samples':int((idx==i).sum()),'mean_confidence':float(conf[idx==i].mean()),'accuracy':float(correct[idx==i].mean())} for i in range(10) if (idx==i).any()]); cal.to_csv(REPORTS/'confidence_calibration.csv',index=False)
fig,a=plt.subplots(figsize=(6.5,6)); a.plot([0,1],[0,1],'--',color='gray',label='Perfect'); a.plot(cal.mean_confidence,cal.accuracy,'o-',label='Model'); a.set(xlim=(0,1),ylim=(0,1),xlabel='Mean confidence',ylabel='Observed accuracy',title='Test confidence calibration'); a.xaxis.set_major_formatter(PercentFormatter(1)); a.yaxis.set_major_formatter(PercentFormatter(1)); a.legend(); fig.tight_layout(); fig.savefig(FIGURES/'confidence_calibration.png',dpi=220,bbox_inches='tight'); plt.show()
print('Graphs:', FIGURES); print('CSV reports:', REPORTS)

NameError: name 'plt' is not defined